In [7]:
for i in range(120, 140):
    byte = bytes([i])
    try:
        char = byte.decode('utf-8')
        print(f"{i:3d} (0x{i:02X}) -> {repr(char)}")
    except UnicodeDecodeError:
        replaced = byte.decode('utf-8', errors='replace')
        print(f"{i:3d} (0x{i:02X}) -> {repr(replaced)} (invalid)")

120 (0x78) -> 'x'
121 (0x79) -> 'y'
122 (0x7A) -> 'z'
123 (0x7B) -> '{'
124 (0x7C) -> '|'
125 (0x7D) -> '}'
126 (0x7E) -> '~'
127 (0x7F) -> '\x7f'
128 (0x80) -> '�' (invalid)
129 (0x81) -> '�' (invalid)
130 (0x82) -> '�' (invalid)
131 (0x83) -> '�' (invalid)
132 (0x84) -> '�' (invalid)
133 (0x85) -> '�' (invalid)
134 (0x86) -> '�' (invalid)
135 (0x87) -> '�' (invalid)
136 (0x88) -> '�' (invalid)
137 (0x89) -> '�' (invalid)
138 (0x8A) -> '�' (invalid)
139 (0x8B) -> '�' (invalid)


Python 在使用 errors='replace' 时会把非法字节替换成 U+FFFD，而不是报错。
这样：
 - 合法部分正常显示
 - 非法字节（如截断、模型生成错误）→ 显示为 ``
 - 程序不会崩溃，用户也能知道“这里有异常”

## array.array

In [3]:
import array

arr = array.array("H", [3, 125, 65535])
print(arr[0])
print(arr.itemsize)

3
2


In [ ]:
arr.append(-1)     # 报错：OverflowError (不能存负数)

OverflowError: unsigned short is less than minimum

In [5]:
arr.append(65536)  # 报错：OverflowError (超出范围)

OverflowError: unsigned short is greater than maximum

In [7]:
import array
import sys

data = list(range(10000))
arr = array.array("H", range(10000)) # 注意：如果数据超过 65535，"H" 会报错，这里假设数据在范围内

# 获取容器本身的大小 (不包含元素对象)
list_size = sys.getsizeof(data)
arr_size = sys.getsizeof(arr)

# 估算总内存 (List 需要加上所有 int 对象的大小)
# 在 64 位 Python 中，小整数对象通常复用，但大整数或新创建的对象会占用额外内存
# 这里简单对比容器 + 元素原始数据
print(f"List 容器大小：{list_size} 字节")
print(f"Array 容器大小：{arr_size} 字节, arr/list: {arr_size/list_size:.2f}")
print(f"Array 元素数据总大小：{len(arr) * arr.itemsize} 字节")



List 容器大小：80056 字节
Array 容器大小：20250 字节, arr/list: 0.25
Array 元素数据总大小：20000 字节


## npy-file

In [8]:
import array
import numpy as np

token_ids_buf = array.array("H")
token_ids_buf.extend([98, 289, 234, 256])
token_ids = np.frombuffer(token_ids_buf, dtype=np.uint16)
np.save("token_ids.npy", token_ids)

In [9]:
token_ids_loaded = np.load("token_ids.npy", mmap_mode="r")
print(token_ids_loaded)

[ 98 289 234 256]


## scatter

In [34]:
import torch

A = torch.zeros((2,3), dtype=torch.int64)
labels = [10, 20]

src = torch.tensor(labels).unsqueeze(0)
print(src, src.dtype)

A.scatter(dim=1, index=torch.tensor([[0, 1]]), src=src)

tensor([[10, 20]]) torch.int64


tensor([[10, 20,  0],
        [ 0,  0,  0]])

In [37]:
A.scatter(dim=1, index=torch.tensor([[1], [2]]), src=torch.tensor(labels).unsqueeze(1))

tensor([[ 0, 10,  0],
        [ 0,  0, 20]])

## top-p Sample

In [44]:
import torch
import torch.nn.functional as  F

logits = torch.tensor([[1, 3, 5, 7, 9], [1, 4, 6, 8, 12]], dtype=torch.float32)
print("logits: ", logits)

logits_sorted, sorted_indices = torch.sort(logits, dim=-1, descending=True)
print("logits_sorted: ", logits_sorted)
print("sorted_indices: ", sorted_indices)
probs = F.softmax(logits_sorted, dim=-1)
print("probs: ", probs)

probs_cursum = torch.cumsum(probs, dim=-1)
print("累计概率: ", probs_cursum)

top_p = 0.999

probs_cursum > top_p

logits:  tensor([[ 1.,  3.,  5.,  7.,  9.],
        [ 1.,  4.,  6.,  8., 12.]])
logits_sorted:  tensor([[ 9.,  7.,  5.,  3.,  1.],
        [12.,  8.,  6.,  4.,  1.]])
sorted_indices:  tensor([[4, 3, 2, 1, 0],
        [4, 3, 2, 1, 0]])
probs:  tensor([[8.6470e-01, 1.1702e-01, 1.5838e-02, 2.1434e-03, 2.9008e-04],
        [9.7929e-01, 1.7936e-02, 2.4274e-03, 3.2852e-04, 1.6356e-05]])
累计概率:  tensor([[0.8647, 0.9817, 0.9976, 0.9997, 1.0000],
        [0.9793, 0.9972, 0.9997, 1.0000, 1.0000]])


tensor([[False, False, False,  True,  True],
        [False, False,  True,  True,  True]])

代码解读：
-   这行代码的意思是：**把"移除标记"向后推迟一位。**
-   原标记：`[False, False, True, True]` (索引 0, 1, 2, 3)
-   取前 N-1 位：`[False, False, True]` (索引 0, 1, 2 的状态)
-   赋值给后 N-1 位 (索引 1, 2, 3)：
        -   索引 1 变成 索引 0 的状态 (`False`)
        -   索引 2 变成 索引 1 的状态 (`False`) <-- **看！索引 2 被救回来了**
        -   索引 3 变成 索引 2 的状态 (`True`)
-   **移位后结果**：`[原索引 0, False, False, True]`
-   **逻辑含义**：如果"你"让累积概率超标了，那么\*\*"你"保留，但"你后面的那个"要移除\*\*。

保护第一个token：

-   **极端情况**：假设 `top_p` 设置得非常小（比如 0.1），而第一个 token 的概率就是 0.5。
-   第一步计算后，索引 0 就会变成 `True` (0.5 > 0.1)。
-   第二步移位后，索引 0 的值取决于它前面的值（不存在），或者保持原样。
-   **安全兜底**：无论阈值多小，**概率最大的那个 token 必须保留**，否则模型就没东西可选了。这行代码强制保证第一个 token 永远不被移除。

性能解读：

**原因 1：GPU 并行加速 (向量化)**

-   Python 的 `for` 循环在 GPU 上是极慢的，因为它无法并行。
-   大模型推理时，Batch Size 可能很大，词表大小 (Vocab Size) 可能是 32000 或 100000+。
-   使用 Tensor 的切片操作 (`[..., 1:]`) 可以在 GPU 上**一次性并行处理所有 Batch 和所有 Token**，速度比循环快几个数量级。

**原因 2：避免分支预测失败**

-   循环中带有 `if` 判断，在并行计算中会导致线程分歧 (Thread Divergence)，降低硬件效率。
-   纯 Tensor 运算（掩码操作）是硬件最喜欢的指令流。

**原因 3：兼容性**

-   这种写法是 PyTorch/TensorFlow 的标准"张量技巧"，虽然人读起来累，但机器执行效率极高。


In [45]:
sorted_indices_to_remove = probs_cursum > top_p
print(sorted_indices_to_remove[..., 1:])
print(sorted_indices_to_remove[..., :-1])
sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
print(sorted_indices_to_remove)
sorted_indices_to_remove[..., 0] = False
print(sorted_indices_to_remove)

tensor([[False, False,  True,  True],
        [False,  True,  True,  True]])
tensor([[False, False, False,  True],
        [False, False,  True,  True]])
tensor([[False, False, False, False,  True],
        [False, False, False,  True,  True]])
tensor([[False, False, False, False,  True],
        [False, False, False,  True,  True]])


In [ ]:
print(sorted_indices.shape, sorted_indices_to_remove.shape, sorted_indices_to_remove.dtype)
indices_to_remove = sorted_indices_to_remove.scatter(dim=-1, index=sorted_indices, src=sorted_indices_to_remove)
# logit经过sort排序，实际要处理的是 未排序的 logit，需要转换为 正确的 index
print(indices_to_remove)

torch.Size([2, 5]) torch.Size([2, 5]) torch.bool
tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False]])


In [49]:
logits[indices_to_remove] = torch.finfo(logits.dtype).min
print(logits)
probs = F.softmax(logits, dim=-1)
print(probs)

tensor([[-3.4028e+38,  3.0000e+00,  5.0000e+00,  7.0000e+00,  9.0000e+00],
        [-3.4028e+38, -3.4028e+38,  6.0000e+00,  8.0000e+00,  1.2000e+01]])
tensor([[0.0000, 0.0021, 0.0158, 0.1171, 0.8650],
        [0.0000, 0.0000, 0.0024, 0.0179, 0.9796]])


In [ ]:
next_token = torch.multinomial(probs, num_samples=1)
# 多运行几次，极大概率是4，低概率是其它
next_token

tensor([[4],
        [3]])

In [59]:
next_token.squeeze(-1)

tensor([4, 3])

### 完整代码

主要错误分析

1\. `scatter` 的维度错误 (严重)

-   **代码:** `indices_to_remove = sorted_indices_to_remove.scatter(0, sorted_indices, sorted_indices_to_remove)`
-   **问题:** `torch.sort` 默认在最后一个维度 (`dim=-1`) 上进行排序。因此 `sorted_indices` 包含的是最后一个维度上的索引。然而，`scatter` 函数却指定了 `dim=0`。
        -   如果 `logits` 是 1D (例如 `[vocab_size]`)，`dim=0` 碰巧是正确的。
        -   如果 `logits` 是 2D (例如 `[batch_size, vocab_size]`)，`dim=0` 对应的是 batch 维度。此时 `sorted_indices` 中的值（0 到 vocab\_size-1）会远超 batch\_size，导致 **`IndexError: index out of range`**。
-   **修正:** 应将 `dim=0` 改为 `dim=-1`，以匹配排序的维度。

2\. Temperature 为 0 时的除零错误 (严重)

-   **代码:** `logits = logits / temperature`
-   **问题:** 如果调用时传入 `temperature=0`，会触发 **`ZeroDivisionError`** 或产生 `inf`。通常 `temperature=0` 意味着贪婪采样（取 Argmax），需要特殊处理。
-   **修正:** 增加对 `temperature` 的判断，若为 0 则直接返回 `argmax`。

3\. 原地修改可能导致的梯度错误 (隐患)

-   **代码:** `logits[indices_to_remove] = float('-inf')`
-   **问题:** 如果输入的 `logits` 张量需要计算梯度 (`requires_grad=True`)，PyTorch 不允许对这样的张量进行原地修改（in-place operation），会报错 **`RuntimeError: a leaf Variable that requires grad is being used in an in-place operation`**。虽然采样通常在 `torch.no_grad()` 下进行，但为了函数健壮性，应避免原地修改。
-   **修正:** 先克隆 `logits` 或确保在 `no_grad` 上下文中使用。

4\. 返回值形状不一致 (体验问题)

-   **代码:** `next_token = torch.multinomial(probs, num_samples=1)`
-   **问题:** `multinomial` 返回的形状是 `(..., 1)`。例如输入是 1D，输出是 `(1, 1)`；输入是 2D，输出是 `(batch, 1)`。通常采样函数期望返回.squeeze() 后的 token ID 形状 `(...,)`。
-   **修正:** 使用 `.squeeze(-1)`。

5\. 设备一致性 (隐患)

-   **代码:** `logits[indices_to_remove] = float('-inf')`
-   **问题:** `float('-inf')` 是 Python 原生浮点数。虽然 PyTorch 通常能自动处理设备转移，但在某些极端版本或混合精度下，显式使用张量常量更安全。
-   **修正:** 使用 `torch.finfo(logits.dtype).min` 或确保常量在正确设备上。

In [66]:
import torch
import torch.nn.functional as F

def sample(logits, temperature=0.7, top_p=0.9):
    # 0. 处理 Temperature 为 0 的情况 (贪婪采样)
    if temperature == 0:
        return torch.argmax(logits, dim=-1, keepdim=False) # 根据需求决定是否 keepdim

    # 1. Temperature 缩放 (克隆以避免原地修改梯度图)
    logits = logits / temperature
    
    # 2. Top-P 过滤 (Nucleus Sampling)
    # 确保在最后一个维度排序
    sorted_logits, sorted_indices = torch.sort(logits, dim=-1, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    
    # 移除累积概率超过 p 的 token
    sorted_indices_to_remove = cumulative_probs > top_p
    
    # 保证至少保留第一个 token (最可能的那个)
    # 移位操作：如果前一个 token 被标记为移除，当前 token 也移除；
    # 但如果前一个 token 没被移除（即使当前 token 累积概率超标），当前 token 保留（为了凑够 top_p 的质量）
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = False # 显式使用 False 而非 0
    
    # 【修正点 1】scatter 的维度必须是 -1 (对应 vocab 维度)，而不是 0
    indices_to_remove = sorted_indices_to_remove.scatter(dim=-1, index=sorted_indices, src=sorted_indices_to_remove)
    
    # 【修正点 5】使用张量常量，确保 dtype 和 device 一致
    logits[indices_to_remove] = torch.finfo(logits.dtype).min
    
    # 3. 计算 Softmax 概率
    probs = F.softmax(logits, dim=-1)
    
    # 4. 采样 (取 1 个样本)
    # 【修正点 4】squeeze 掉最后一个维度，使返回形状更直观
    next_token = torch.multinomial(probs, num_samples=1).squeeze(-1)
    
    return next_token

logits = torch.tensor([[1, 3, 10, 11, 9], [1, 14, 20, 8, 12]], dtype=torch.float32)
print(sample(logits.clone()))
print(sample(logits.clone(), temperature=0))


tensor([2, 2])
tensor([3, 2])
